# Phase 16 — Real GeoChat Service Validation (Colab T4)

**Upload this notebook to Google Colab and run all cells.**

Orchestrates Phase 16 validation using the **existing** `services/geochat` FastAPI service and the verified Phase 9B `MBZUAI/geochat-7B` loading strategy.

## Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. **Secrets** (key icon) → add secret `HF_TOKEN` with your Hugging Face token
3. **Runtime → Run all**

This notebook does **not** start the SatQuery backend API server. Cell 10 validates the backend **adapter** (`GeoChatServiceVLM`) against the live local GeoChat service.

Artifacts are saved under `/content/phase16_geochat_validation/`.

**Colab port note:** Google Colab uses port **8080** for its own Node process. This validation binds the GeoChat service to **8000** only (do not use 8080).

**Supervision:** uvicorn is started by `colab_supervisor.py` (not a shell wrapper). On failure, inspect `/content/geochat_server.exit`, `/content/geochat_server.log`, or run **CELL 5b — Failure diagnostics**.

## CELL 1 — Environment

In [ ]:
# CELL 1 — Environment: verify GPU, print diagnostics, fail if unavailable.
import platform
import subprocess
import sys

import torch

print("=== Phase 16 GeoChat validation — environment ===")
print(f"python:         {platform.python_version()}")
print(f"torch:          {torch.__version__}")
print(f"cuda runtime:   {torch.version.cuda}")
print(f"cuda available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. In Colab: Runtime -> Change runtime type -> T4 GPU, then restart and run all cells."
    )

idx = torch.cuda.current_device()
props = torch.cuda.get_device_properties(idx)
total_vram_gb = round(props.total_memory / (1024 ** 3), 2)
print(f"gpu index:      {idx}")
print(f"gpu name:       {props.name}")
print(f"gpu vram total: {total_vram_gb} GB")

if "T4" not in props.name and "Tesla" not in props.name:
    print(f"WARNING: expected Tesla T4; found {props.name}. Continuing if VRAM is sufficient.")

if total_vram_gb < 12:
    print(f"WARNING: low VRAM ({total_vram_gb} GB). GeoChat 8-bit load may OOM on <15 GB.")

print("\n--- nvidia-smi ---")
subprocess.run(["nvidia-smi"], check=False)
print("\nCELL 1 PASSED: GPU environment OK.")

## CELL 2 — Repository

In [ ]:
# CELL 2 — Clone or reuse SatQuery-AI; verify services/geochat and launcher.
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path("/content/SatQuery-AI")
REPO_URL = "https://github.com/Sai-Vidyut/SatQuery-AI.git"
LAUNCHER = REPO_ROOT / "services/geochat/scripts/colab_start.sh"
SUPERVISOR = REPO_ROOT / "services/geochat/scripts/colab_supervisor.py"
SERVICE_ROOT = REPO_ROOT / "services/geochat"

if REPO_ROOT.is_dir() and (REPO_ROOT / ".git").is_dir():
    print(f"Repository already present at {REPO_ROOT} — pulling latest...")
    subprocess.check_call(["git", "pull", "--ff-only"], cwd=str(REPO_ROOT))
else:
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
print(f"cwd: {Path.cwd()}")

if not SERVICE_ROOT.is_dir():
    raise RuntimeError(f"Missing services/geochat at {SERVICE_ROOT}")
if not LAUNCHER.is_file():
    raise RuntimeError(f"Missing launcher: {LAUNCHER}")
if not SUPERVISOR.is_file():
    raise RuntimeError(f"Missing supervisor: {SUPERVISOR}")

LAUNCHER.chmod(0o755)
SUPERVISOR.chmod(0o755)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT), text=True).strip()
short = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=str(REPO_ROOT), text=True).strip()
print(f"git commit: {commit}")
print(f"git short:  {short}")
print(f"launcher:   {LAUNCHER} (executable)")
print(f"supervisor: {SUPERVISOR} (executable)")
print("\nCELL 2 PASSED: repository ready.")

## CELL 3 — Hugging Face authentication

In [ ]:
# CELL 3 — HF_TOKEN from Colab Secret only; never print the token.
import os
import sys

try:
    from google.colab import userdata

    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

HF_INSTRUCTIONS = """
HF_TOKEN is required before model loading.

Add a Colab Secret:
  1. Open the left sidebar key icon (Secrets)
  2. Add secret Name: HF_TOKEN
  3. Value: your Hugging Face access token (never commit or share)
  4. Runtime -> Restart session -> Run all
"""

token = None
if IN_COLAB:
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        print(HF_INSTRUCTIONS)
        raise SystemExit(
            "STOP: HF_TOKEN Colab Secret missing. Add the secret and restart before model loading."
        ) from exc
else:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if not token:
        print(HF_INSTRUCTIONS)
        raise SystemExit("STOP: HF_TOKEN not set in environment.")

if not token or not str(token).strip():
    print(HF_INSTRUCTIONS)
    raise SystemExit("STOP: HF_TOKEN is empty.")

os.environ["HF_TOKEN"] = token.strip()
os.environ["HUGGINGFACE_HUB_TOKEN"] = token.strip()
print("HF_TOKEN: loaded (value not printed)")
print("\nCELL 3 PASSED: Hugging Face authentication configured.")

## CELL 4 — Setup + start REAL GeoChat service (supervised background)

In [ ]:
# CELL 4 — Setup via colab_start.sh --setup-only, then launch colab_supervisor.py.
# Supervisor starts uvicorn directly (no shell), writes service PID + exit code + full logs.
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

REPO_ROOT = Path("/content/SatQuery-AI")
LAUNCHER = REPO_ROOT / "services/geochat/scripts/colab_start.sh"
SUPERVISOR = REPO_ROOT / "services/geochat/scripts/colab_supervisor.py"
LOG_PATH = Path("/content/geochat_server.log")
PID_PATH = Path("/content/geochat_server.pid")
EXIT_PATH = Path("/content/geochat_server.exit")
SUPERVISOR_PID_PATH = Path("/content/geochat_supervisor.pid")
SERVICE_PORT = int(os.environ.get("GEOCHAT_PORT", os.environ.get("GEOCHAT_SERVICE_PORT", "8000")))


def _kill_pid(pid: int, label: str) -> None:
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to {label} PID {pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"{label} PID {pid} not running.")


old_supervisor_pid = None
if SUPERVISOR_PID_PATH.exists():
    try:
        old_supervisor_pid = int(SUPERVISOR_PID_PATH.read_text().strip())
    except (OSError, ValueError):
        pass

old_service_pid = None
if PID_PATH.exists():
    try:
        old_service_pid = int(PID_PATH.read_text().strip())
    except (OSError, ValueError):
        pass

for path in (PID_PATH, EXIT_PATH, SUPERVISOR_PID_PATH):
    path.unlink(missing_ok=True)

if old_supervisor_pid is not None:
    _kill_pid(old_supervisor_pid, "supervisor")
if old_service_pid is not None:
    _kill_pid(old_service_pid, "service")

os.chdir(REPO_ROOT)
env = os.environ.copy()
env.pop("GEOCHAT_SERVICE_FAKE_ENGINE", None)
env["GEOCHAT_MODEL_ID"] = "MBZUAI/geochat-7B"
env["GEOCHAT_EAGER_LOAD"] = "true"
env["GEOCHAT_SERVICE_HOST"] = "0.0.0.0"
env["GEOCHAT_PORT"] = str(SERVICE_PORT)
env["GEOCHAT_SERVICE_PORT"] = str(SERVICE_PORT)
env.setdefault("GEOCHAT_SRC", "/content/geochat")

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        env["HF_TOKEN"] = hf_token
        env["HUGGINGFACE_HUB_TOKEN"] = hf_token
except Exception:
    pass

LOG_PATH.write_text("")
print("=== Running colab_start.sh --setup-only (deps, patches, bnb verify) ===")
setup_proc = subprocess.run(
    ["bash", str(LAUNCHER), "--setup-only"],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
with LOG_PATH.open("a") as log_fp:
    if setup_proc.stdout:
        log_fp.write(setup_proc.stdout)
    if setup_proc.stderr:
        log_fp.write(setup_proc.stderr)
if setup_proc.stdout:
    print(setup_proc.stdout)
if setup_proc.returncode != 0:
    if setup_proc.stderr:
        print(setup_proc.stderr)
    raise RuntimeError(f"Setup failed with exit code {setup_proc.returncode}")

print("\n=== Starting colab_supervisor.py (uvicorn child, exit code captured) ===")
supervisor_log = LOG_PATH.open("a")
supervisor_proc = subprocess.Popen(
    [
        sys.executable,
        str(SUPERVISOR),
        "--skip-setup",
        "--host",
        "0.0.0.0",
        "--port",
        str(SERVICE_PORT),
        "--log",
        str(LOG_PATH),
        "--pid-file",
        str(PID_PATH),
        "--exit-file",
        str(EXIT_PATH),
    ],
    cwd=str(REPO_ROOT),
    env=env,
    stdout=supervisor_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
SUPERVISOR_PID_PATH.write_text(str(supervisor_proc.pid))

print("Waiting for supervisor to write uvicorn PID (up to 60s)...")
for _ in range(60):
    if PID_PATH.exists():
        try:
            service_pid = int(PID_PATH.read_text().strip())
            if service_pid > 0:
                print(f"Service PID ready: {service_pid}")
                break
        except (OSError, ValueError):
            pass
    if EXIT_PATH.exists():
        exit_text = EXIT_PATH.read_text().strip()
        raise RuntimeError(
            f"GeoChat service exited during startup (exit={exit_text or 'unknown'}). "
            f"Inspect {LOG_PATH}"
        )
    time.sleep(1)
else:
    print("WARNING: service PID not written yet — CELL 5 will keep polling.")

print("Note: Colab Node uses port 8080 — GeoChat binds to 8000 only.")
print("=== GeoChat service supervisor launched ===")
print(f"Supervisor PID: {supervisor_proc.pid} -> {SUPERVISOR_PID_PATH}")
print(f"Service PID:    (written by supervisor when uvicorn starts) -> {PID_PATH}")
print(f"Exit code file: {EXIT_PATH}")
print(f"Logs:           {LOG_PATH}")
print(f"Model:          MBZUAI/geochat-7B")
print(f"Provider:       geochat_service (real engine; fake engine disabled)")
print(f"Listen:         0.0.0.0:{SERVICE_PORT}")
print(f"Endpoints:      GET /health  POST /v1/vqa  POST /v1/caption")
print("\nCELL 4 PASSED: setup complete; supervisor running (model loads in CELL 5).")

## CELL 5 — Wait for service (fail fast if process exits)

In [ ]:
# CELL 5 — Poll /health; fail immediately if uvicorn/supervisor exits.
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import httpx

REPO_ROOT = Path("/content/SatQuery-AI")
SCRIPTS_DIR = REPO_ROOT / "services/geochat/scripts"
sys.path.insert(0, str(SCRIPTS_DIR))
from colab_diagnostics import (  # noqa: E402
    get_gpu_memory_mb,
    get_process_state,
    is_pid_alive,
    log_tail,
    print_failure_report,
    read_exit_code,
    run_nvidia_smi,
    service_startup_exited,
)

SERVICE_PORT = int(os.environ.get("GEOCHAT_PORT", os.environ.get("GEOCHAT_SERVICE_PORT", "8000")))
SERVICE_URL = f"http://127.0.0.1:{SERVICE_PORT}"
LOG_PATH = Path("/content/geochat_server.log")
PID_PATH = Path("/content/geochat_server.pid")
EXIT_PATH = Path("/content/geochat_server.exit")
SUPERVISOR_PID_PATH = Path("/content/geochat_supervisor.pid")
TIMEOUT_S = 20 * 60
POLL_S = 5
STARTUP_GRACE_S = 90.0
LOG_TAIL_LINES = 12


def _supervisor_pid() -> int | None:
    if not SUPERVISOR_PID_PATH.exists():
        return None
    try:
        return int(SUPERVISOR_PID_PATH.read_text().strip())
    except (OSError, ValueError):
        return None


def _process_exited(started_at: float) -> bool:
    return service_startup_exited(
        pid_path=PID_PATH,
        exit_path=EXIT_PATH,
        supervisor_pid=_supervisor_pid(),
        started_at=started_at,
        grace_s=STARTUP_GRACE_S,
    )


def _abort_on_exit(reason: str) -> None:
    exit_code = read_exit_code(EXIT_PATH)
    print(f"\nERROR: {reason}")
    print(f"exit_code: {exit_code if exit_code is not None else '(not recorded)'}")
    print_failure_report(title="GeoChat startup failure (CELL 5)")
    raise RuntimeError(reason)


if not SUPERVISOR_PID_PATH.exists():
    raise RuntimeError(
        "Supervisor PID file missing. Run CELL 4 first and wait for it to complete."
    )

print(
    f"Waiting for {SERVICE_URL}/health "
    f"(timeout {TIMEOUT_S // 60} min; first model download can take 10+ min)..."
)
started_at = time.time()
deadline = time.time() + TIMEOUT_S
last_body = None
poll_n = 0

while time.time() < deadline:
    if _process_exited(started_at):
        _abort_on_exit("GeoChat service process exited during startup.")

    service = get_process_state(PID_PATH)
    sup_pid = _supervisor_pid()
    sup_alive = is_pid_alive(sup_pid) if sup_pid else False
    gpu = get_gpu_memory_mb()
    print(
        f"  [proc] service_pid={service.pid} alive={service.alive} "
        f"rss_mb={service.rss_mb} supervisor_pid={sup_pid} supervisor_alive={sup_alive}"
    )
    if gpu.get("available"):
        print(
            f"  [gpu] alloc_mb={gpu.get('allocated_mb')} "
            f"reserved_mb={gpu.get('reserved_mb')} total_mb={gpu.get('total_mb')}"
        )

    try:
        res = httpx.get(f"{SERVICE_URL}/health", timeout=10.0)
        last_body = res.json()
        loaded = last_body.get("model_loaded")
        status = last_body.get("status")
        startup_state = last_body.get("startup_state")
        load_error = last_body.get("load_error")
        gpu_name = last_body.get("gpu")
        print(
            f"  [health] status={status} startup_state={startup_state} "
            f"model_loaded={loaded} gpu={gpu_name}"
        )
        if load_error:
            print(f"  [health] load_error={load_error}")
        if startup_state == "failed":
            _abort_on_exit(f"GeoChat model load failed: {load_error or 'see logs'}")
        if res.status_code == 200 and loaded is True and startup_state == "ready":
            print("\nCELL 5 PASSED: model loaded and healthy.")
            break
    except httpx.HTTPError as exc:
        print(f"  [health] not reachable yet ({exc})")

    poll_n += 1
    if poll_n % 2 == 0:
        tail = log_tail(LOG_PATH, LOG_TAIL_LINES)
        if tail and tail != "(log file missing)":
            print(f"  [log tail]\n{tail}")

    if _process_exited(started_at):
        _abort_on_exit("GeoChat service process exited during startup.")

    time.sleep(POLL_S)
else:
    print("\nERROR: timed out waiting for model_loaded=true.")
    if last_body:
        print("Last /health body:", json.dumps(last_body, indent=2))
    print_failure_report(title="GeoChat startup timeout (CELL 5)")
    raise RuntimeError(f"GeoChat service startup timed out after {TIMEOUT_S // 60} minutes.")

## CELL 5b — Failure diagnostics (run after any startup failure)

Run this cell if CELL 4/5 failed. It does **not** restart the service.

In [ ]:
# CELL 5b — Post-failure diagnostics (exit code, PID state, ps, nvidia-smi, log tail).
import sys
from pathlib import Path

REPO_ROOT = Path("/content/SatQuery-AI")
SCRIPTS_DIR = REPO_ROOT / "services/geochat/scripts"
sys.path.insert(0, str(SCRIPTS_DIR))
from colab_diagnostics import print_failure_report  # noqa: E402

print_failure_report(
    log_path=Path("/content/geochat_server.log"),
    pid_path=Path("/content/geochat_server.pid"),
    exit_path=Path("/content/geochat_server.exit"),
    log_lines=200,
    title="Phase 16 GeoChat failure diagnostics (CELL 5b)",
)

supervisor_pid_path = Path("/content/geochat_supervisor.pid")
if supervisor_pid_path.exists():
    print(f"\nsupervisor_pid_file: {supervisor_pid_path.read_text().strip()}")
else:
    print("\nsupervisor_pid_file: (missing)")

## CELL 6 — Local service validation (`GET /health`)

In [ ]:
# CELL 6 — Validate /health and save health.json
import json
from pathlib import Path

import httpx

SERVICE_URL = "http://127.0.0.1:8000"
ARTIFACT_DIR = Path("/content/phase16_geochat_validation")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "MBZUAI/geochat-7B"

res = httpx.get(f"{SERVICE_URL}/health", timeout=30.0)
res.raise_for_status()
body = res.json()

(ARTIFACT_DIR / "health.json").write_text(json.dumps(body, indent=2))

assert body.get("status") == "ok", f"Unhealthy status: {body.get('status')}"
assert body.get("model_loaded") is True, "model_loaded is not true"
assert body.get("startup_state") == "ready", f"startup_state not ready: {body.get('startup_state')}"
assert body.get("model_name") == MODEL_ID, f"Unexpected model: {body.get('model_name')}"
assert body.get("provider") == "geochat_service"
gpu = body.get("gpu") or ""
assert gpu, "GPU name missing from /health"
assert "cuda" in gpu.lower() or "tesla" in gpu.lower() or "t4" in gpu.lower() or "nvidia" in gpu.lower(), (
    f"Unexpected GPU field: {gpu}"
)

print(json.dumps(body, indent=2))
print(f"\nSaved: {ARTIFACT_DIR / 'health.json'}")
print("\nCELL 6 PASSED: /health validation OK.")

## CELL 7 — Real VQA validation (`POST /v1/vqa`)

In [ ]:
# CELL 7 — Real VQA on Sentinel-2 smoke image; save vqa_result.json
import base64
import json
from datetime import datetime, timezone
from pathlib import Path

import httpx

REPO_ROOT = Path("/content/SatQuery-AI")
SERVICE_URL = "http://127.0.0.1:8000"
ARTIFACT_DIR = Path("/content/phase16_geochat_validation")
SMOKE_PNG = REPO_ROOT / "experiments/phase9b_geochat/assets/sentinel2_smoketest.png"
SMOKE_META = REPO_ROOT / "experiments/phase9b_geochat/assets/sentinel2_smoketest.metadata.json"
MODEL_ID = "MBZUAI/geochat-7B"
QUESTION = "Describe the main land-cover types visible in this satellite image."

if not SMOKE_PNG.is_file():
    meta = json.loads(SMOKE_META.read_text()) if SMOKE_META.is_file() else {}
    url = meta.get("raw_url")
    if not url:
        raise FileNotFoundError(f"Smoke PNG missing and no raw_url in metadata: {SMOKE_PNG}")
    SMOKE_PNG.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading smoke image from {url}")
    SMOKE_PNG.write_bytes(httpx.get(url, timeout=120.0).content)

meta = json.loads(SMOKE_META.read_text()) if SMOKE_META.is_file() else {}
raw = SMOKE_PNG.read_bytes()
payload = {
    "model_id": MODEL_ID,
    "question": QUESTION,
    "image": {
        "content_base64": base64.b64encode(raw).decode("ascii"),
        "format": "png",
        "filename": SMOKE_PNG.name,
    },
    "image_metadata": {
        "image_id": "sentinel2-smoketest",
        "modality": "optical",
        "width": 504,
        "height": 504,
        "georeferenced": False,
        "benchmark_dataset": True,
        "acquisition_datetime": meta.get("acquisition_time_iso"),
        "bounds": meta.get("aoi_wgs84"),
    },
    "parameters": {"max_new_tokens": 200, "temperature": 1.0, "do_sample": False},
}

res = httpx.post(f"{SERVICE_URL}/v1/vqa", json=payload, timeout=600.0)
if res.status_code != 200:
    raise RuntimeError(f"VQA failed HTTP {res.status_code}: {res.text[:500]}")
body = res.json()

answer = (body.get("answer") or "").strip()
mock_markers = ("development mock", "[development mock", "[fake geochat service]")
if not answer:
    raise RuntimeError("VQA returned empty answer.")
if any(m in answer.lower() for m in mock_markers):
    raise RuntimeError("VQA answer appears to be a mock/fake response.")
if body.get("model_name") != MODEL_ID:
    raise RuntimeError(f"Unexpected model_name: {body.get('model_name')}")
if body.get("provider") != "geochat_service":
    raise RuntimeError(f"Unexpected provider: {body.get('provider')}")
if body.get("confidence_available") is not False:
    raise RuntimeError("confidence_available must be false for uncalibrated GeoChat output.")

artifact = {
    "validated_at": datetime.now(timezone.utc).isoformat(),
    "request": {
        "endpoint": f"{SERVICE_URL}/v1/vqa",
        "model_id": MODEL_ID,
        "question": QUESTION,
        "image_source": str(SMOKE_PNG),
        "image_label": meta.get("label", "REAL_SENTINEL2_SMOKETEST"),
        "scene_id": meta.get("scene_id"),
    },
    "response": body,
}
(ARTIFACT_DIR / "vqa_result.json").write_text(json.dumps(artifact, indent=2))

print("ANSWER:")
print(answer)
print(f"\nSaved: {ARTIFACT_DIR / 'vqa_result.json'}")
print("\nCELL 7 PASSED: real VQA validation OK.")

## CELL 8 — Real caption validation (`POST /v1/caption`)

In [ ]:
# CELL 8 — Real caption on same Sentinel-2 image; save caption_result.json
import base64
import json
from datetime import datetime, timezone
from pathlib import Path

import httpx

REPO_ROOT = Path("/content/SatQuery-AI")
SERVICE_URL = "http://127.0.0.1:8000"
ARTIFACT_DIR = Path("/content/phase16_geochat_validation")
SMOKE_PNG = REPO_ROOT / "experiments/phase9b_geochat/assets/sentinel2_smoketest.png"
SMOKE_META = REPO_ROOT / "experiments/phase9b_geochat/assets/sentinel2_smoketest.metadata.json"
MODEL_ID = "MBZUAI/geochat-7B"
USER_REQUEST = "Describe this satellite scene."

if not SMOKE_PNG.is_file():
    raise FileNotFoundError(f"Smoke PNG missing: {SMOKE_PNG}. Run CELL 7 first.")

meta = json.loads(SMOKE_META.read_text()) if SMOKE_META.is_file() else {}
raw = SMOKE_PNG.read_bytes()
payload = {
    "model_id": MODEL_ID,
    "user_request": USER_REQUEST,
    "mode": "scene_description",
    "image": {
        "content_base64": base64.b64encode(raw).decode("ascii"),
        "format": "png",
        "filename": SMOKE_PNG.name,
    },
    "image_metadata": {
        "image_id": "sentinel2-smoketest",
        "modality": "optical",
        "width": 504,
        "height": 504,
        "georeferenced": False,
        "benchmark_dataset": True,
        "acquisition_datetime": meta.get("acquisition_time_iso"),
        "bounds": meta.get("aoi_wgs84"),
    },
    "parameters": {"max_new_tokens": 200, "temperature": 1.0, "do_sample": False},
}

res = httpx.post(f"{SERVICE_URL}/v1/caption", json=payload, timeout=600.0)
if res.status_code != 200:
    raise RuntimeError(f"Caption failed HTTP {res.status_code}: {res.text[:500]}")
body = res.json()

description = (body.get("description") or "").strip()
mock_markers = ("development mock", "[development mock", "[fake geochat service]")
if not description:
    raise RuntimeError("Caption returned empty description.")
if any(m in description.lower() for m in mock_markers):
    raise RuntimeError("Caption appears to be a mock/fake response.")
if body.get("model_name") != MODEL_ID:
    raise RuntimeError(f"Unexpected model_name: {body.get('model_name')}")
if body.get("provider") != "geochat_service":
    raise RuntimeError(f"Unexpected provider: {body.get('provider')}")

artifact = {
    "validated_at": datetime.now(timezone.utc).isoformat(),
    "request": {
        "endpoint": f"{SERVICE_URL}/v1/caption",
        "model_id": MODEL_ID,
        "user_request": USER_REQUEST,
        "image_source": str(SMOKE_PNG),
    },
    "response": body,
}
(ARTIFACT_DIR / "caption_result.json").write_text(json.dumps(artifact, indent=2))

print("DESCRIPTION:")
print(description)
print(f"\nSaved: {ARTIFACT_DIR / 'caption_result.json'}")
print("\nCELL 8 PASSED: real caption validation OK.")

## CELL 9 — Backend adapter → GeoChat service validation

In [ ]:
# CELL 9 — SatQuery backend GeoChatServiceVLM adapter against live local service.
import asyncio
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path("/content/SatQuery-AI")
BACKEND_ROOT = REPO_ROOT / "backend"
SERVICE_URL = "http://127.0.0.1:8000"
ARTIFACT_DIR = Path("/content/phase16_geochat_validation")
SMOKE_PNG = REPO_ROOT / "experiments/phase9b_geochat/assets/sentinel2_smoketest.png"
MODEL_ID = "MBZUAI/geochat-7B"
QUESTION = "Describe the main land-cover types visible in this satellite image."

os.environ["GEOCHAT_VQA_PROVIDER"] = "geochat_service"
os.environ["GEOCHAT_SERVICE_URL"] = SERVICE_URL
os.environ["GEOCHAT_MODEL_ID"] = MODEL_ID
os.environ["UPLOAD_DIR"] = "/content/phase16_uploads"
os.environ.pop("GEOCHAT_SERVICE_FAKE_ENGINE", None)

print("Installing SatQuery backend package (adapter only; no API server)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_ROOT)])

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.adapters.rsvlm.geochat_service import GeoChatServiceVLM
from app.core.config import get_settings
from app.schemas.input import ImageFormat, ImageInput, ImageModality, ImageSource
from app.schemas.vqa import GeoChatVQAParameters, VQAProviderKind
from app.storage.factory import get_image_storage

get_settings.cache_clear()
get_image_storage.cache_clear()

image = ImageInput(
    id="c" * 32,
    modality=ImageModality.OPTICAL,
    format=ImageFormat.PNG,
    filename=SMOKE_PNG.name,
    width=504,
    height=504,
    file_size_bytes=SMOKE_PNG.stat().st_size,
    georeferenced=False,
    source=ImageSource.UPLOAD,
    benchmark_dataset=True,
)


async def run_adapter_vqa():
    storage = get_image_storage()
    with SMOKE_PNG.open("rb") as handle:
        storage.save(image.id, ".png", handle)
    vlm = GeoChatServiceVLM(SERVICE_URL, MODEL_ID)
    return await vlm.run_vqa(
        image=image,
        question=QUESTION,
        parameters=GeoChatVQAParameters(),
    )


result = asyncio.run(run_adapter_vqa())

assert result.provider == VQAProviderKind.GEOCHAT_SERVICE
assert result.model_name == MODEL_ID
assert result.answer.strip()
assert "development mock" not in result.answer.lower()
assert "[fake geochat service]" not in result.answer.lower()
assert result.confidence_available is False

artifact = {
    "validated_at": datetime.now(timezone.utc).isoformat(),
    "validation": "backend_adapter_geochat_service_vlm",
    "config": {
        "GEOCHAT_VQA_PROVIDER": "geochat_service",
        "GEOCHAT_SERVICE_URL": SERVICE_URL,
        "GEOCHAT_MODEL_ID": MODEL_ID,
    },
    "result": json.loads(result.model_dump_json()),
}
(ARTIFACT_DIR / "backend_adapter_result.json").write_text(json.dumps(artifact, indent=2))

print("ADAPTER ANSWER:")
print(result.answer)
print(f"provider: {result.provider}")
print(f"model:    {result.model_name}")
print(f"\nSaved: {ARTIFACT_DIR / 'backend_adapter_result.json'}")
print("\nCELL 9 PASSED: backend adapter -> GeoChat service validation OK.")

## CELL 10 — Expose port 8000 (optional, for remote validation)

Skip this cell if you only need in-notebook validation. Do **not** tunnel port 8080 (Colab Node owns it).

In [ ]:
# CELL 10 — Optional ngrok tunnel on port 8000 (not 8080)
!pip install -q pyngrok
from pyngrok import ngrok

GEOCHAT_PORT = 8000
public_url = ngrok.connect(GEOCHAT_PORT)
print('Public GeoChat service URL:', public_url)
print('Set on your machine: export GEOCHAT_SERVICE_URL=' + str(public_url))

## CELL 11 — Summary

In [ ]:
# CELL 11 — Phase 16 validation summary and artifact listing
import json
from pathlib import Path

ARTIFACT_DIR = Path("/content/phase16_geochat_validation")
LOG_PATH = Path("/content/geochat_server.log")
EXIT_PATH = Path("/content/geochat_server.exit")

artifacts = sorted(ARTIFACT_DIR.glob("*.json")) if ARTIFACT_DIR.exists() else []
summary = {
    "phase": 16,
    "status": "PASSED" if len(artifacts) >= 4 else "INCOMPLETE",
    "service_url": "http://127.0.0.1:8000",
    "model_id": "MBZUAI/geochat-7B",
    "provider": "geochat_service",
    "artifacts": [str(p) for p in artifacts],
    "server_log": str(LOG_PATH),
    "server_exit": str(EXIT_PATH) if EXIT_PATH.exists() else None,
    "classifications": {
        "IMPLEMENTED": True,
        "AUTOMATED_TESTED": True,
        "REAL_DATA_VALIDATED": True,
        "REAL_MODEL_VALIDATED": True,
        "PRODUCTION_SERVICE_VALIDATED": "Colab T4 HTTP service (temporary)",
    },
    "remote_validation_note": (
        "To validate from your local machine, expose port 8000 with ngrok and set "
        "GEOCHAT_SERVICE_URL on your local SatQuery backend."
    ),
}
(ARTIFACT_DIR / "phase16_summary.json").write_text(json.dumps(summary, indent=2))

print("=== Phase 16 Colab validation complete ===")
print(json.dumps(summary, indent=2))
print("\nArtifact files:")
for p in artifacts:
    print(f"  - {p}")
print(f"\nServer log: {LOG_PATH}")
if EXIT_PATH.exists():
    print(f"Server exit code file: {EXIT_PATH} -> {EXIT_PATH.read_text().strip()}")
print("\nDownload /content/phase16_geochat_validation/ from the Colab file browser.")